In [28]:
import os
import numpy as np
import sympy as sp
import torch

from SymbolicMathematics.src.envs import set_cover_utils
from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules

from src.utils import to_cuda
from src.envs.sympy_utils import simplify
from src.envs.set_cover_utils import validate_set_cover_solution

## Build environment / Reload model

In [42]:
# trained model, e.g. "wget https://dl.fbaipublicfiles.com/SymbolicMathematics/models/fwd_bwd.pth"
#model_path = './fwd_bwd.pth'
model_path = './dumped/first_train/kxshkrx981/checkpoint.pth'
assert os.path.isfile(model_path)

In [43]:
params = params = AttrDict({

    # environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    #'tasks': 'prim_fwd',
    'tasks': 'set_cover',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # model parameters
    'cpu': True,
    'emb_dim': 1024,
    'n_enc_layers': 6, #it was 6
    'n_dec_layers': 6, #it was 6
    'n_heads': 8, #it was 8
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path,

})

In [44]:
env = build_env(params)
#x = env.local_dict['x']

In [45]:
modules = build_modules(env, params)
encoder = modules['encoder']
decoder = modules['decoder']

In [46]:
set_instance = [[1, 2, 3], [2, 3, 4], [1, 4, 5]]

In [47]:
def list_to_tokens(lst):
    return [str(c) for c in str(lst) if str(c) != ' ']

In [48]:
instance_tokens = list_to_tokens(set_instance)
print(instance_tokens)

['[', '[', '1', ',', '2', ',', '3', ']', ',', '[', '2', ',', '3', ',', '4', ']', ',', '[', '1', ',', '4', ',', '5', ']', ']']


### Encode input

In [49]:
#x1_prefix = env.clean_prefix(['sub', 'derivative', 'f', 'x', 'x'] + f_prefix)
x1_prefix = instance_tokens
x1 = torch.LongTensor(
    [env.eos_index] +
    [env.word2id[w] for w in x1_prefix] +
    [env.eos_index]
).view(-1, 1)
len1 = torch.LongTensor([len(x1)])
#x1, len1 = to_cuda(x1, len1)

with torch.no_grad():
    encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

### Decode with beam search

In [50]:
beam_size = 10
with torch.no_grad():
    _, _, beam = decoder.generate_beam(encoded, len1, beam_size=beam_size, length_penalty=1.0, early_stopping=1, max_len=200)
    assert len(beam) == 1
hypotheses = beam[0].hyp
assert len(hypotheses) == beam_size

### Print results

In [51]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    return list_of_lists  # Return the final list of lists

In [52]:
#print(f"Input function f: {f}")
#print(f"Reference function F: {F}")
#print("")
print("Problem:",set_instance)
for score, sent in sorted(hypotheses, key=lambda x: x[0], reverse=True):

    # parse decoded hypothesis
    ids = sent[1:].tolist()                  # decoded token IDs
    tok = [env.id2word[wid] for wid in ids]  # convert to prefix

    try:
        #hyp = env.prefix_to_infix(tok)       # convert to infix
        #hyp = env.infix_to_sympy(hyp)        # convert to SymPy
        hyp = convert_token_sequence_to_list_of_lists(tok)

        # check whether we recover f if we differentiate the hypothesis
        # note that sometimes, SymPy fails to show that hyp' - f == 0, and the result is considered as invalid, although it may be correct
        #res = "OK" if simplify(hyp.diff(x) - f, seconds=1) == 0 else "NO"

    except:
        res = "INVALID PREFIX EXPRESSION"
        hyp = tok

    # print result
    print("%.5f  %s" % (score, hyp))

Problem: [[1, 2, 3], [2, 3, 4], [1, 4, 5]]
-0.06784  [[[1, 2, 3], [2, 3, 4, 5], [0, 5]]]
-0.07398  [[[1, 2, 3], [0, 4], [1, 4, 5]]]
-0.11674  [[[1, 2, 3], [0, 4, 5]]]
-0.12500  [[[1, 2, 3], [1, 4, 5], [0, 4, 5]]]
-0.12567  [[[1, 2, 3], [2, 4, 5], [0, 4, 5]]]
-0.16663  [[[1, 2, 3], [2, 3, 4, 5]]]
-0.17230  [[[2, 3, 3], [1, 2, 3, 4], [0, 5]]]
-0.17914  [[[1, 2, 3], [2, 3, 4], [0, 5]]]
-0.17922  [[[1, 2, 3], [2, 3, 4, 5], [0, 4]]]
-0.21610  [[[1, 2, 3], [2, 4], [0, 5]]]
